<div class="alert">Get the residuals from ZTF-sHG1G2 for Dima's method</div>

In [1]:
import os
import time
import sys

sys.path.append("..")


import numpy as np

import ssptools
from astropy.coordinates import SkyCoord
import rocks

from fink_utils.sso.spins import (
    estimate_sso_params,
    func_sshg1g2,
)  # , func_hg1g2, cos_aspect_angle
from fink_utils.sso.periods import estimate_synodic_period
from fink_utils.sso.utils import compute_light_travel_correction, estimate_axes_ratio

import matplotlib.pyplot as plt


# import seaborn as sns
# sns.set_context("poster")

In [2]:
fink_colors = ["#15284F", "#F5622E"]

# Functions to add to fink-utils

In [3]:
def angle_between_vectors(v1, v2):
    """
    Compute the angle between two 3D vectors.

    Parameters
    ----------
    v1 : list or np.ndarray
        The first 3D vector.
    v2 : list or np.ndarray
        The second 3D vector.

    Returns
    -------
    float
        The angle between the two vectors in radians.
    """
    v1 = np.array(v1)
    v2 = np.array(v2)

    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    cos_theta = dot_product / (norm_v1 * norm_v2)
    angle = np.arccos(np.clip(cos_theta, -1.0, 1.0))  # Clip to handle numerical issues

    return angle


# Example usage
# v1 = [1.0, 0, 0]
# v2 = [-1.0, -1.0, 1]
# angle = angle_between_vectors(v1, v2)
# print(f"The angle between the vectors is {np.degrees(angle):.2f} degrees")


def synodic_to_sidereal(synodic_period, X):
    """
    Convert synodic rotation period to sidereal rotation period.

    TBD

    Parameters
    ----------
    synodic_period : float
        Synodic rotation period in days.

    Returns
    -------
    sidereal_period : float
        Sidereal rotation period in days.
    """
    sidereal_period = synodic_period
    return sidereal_period

# Target definition

In [4]:
ssnamenr = 5209
ssnamenr = 136108  # KBO = fixed = Haumea
# ssnamenr = 186153 # High frequency observation
# ssnamenr = 223 # Example article
# ssnamenr = 9799 # JTO lotta obs

# ssnamenr = 17365 # 1978VF11 Thymbareus

ssnamenr = 17365

ssocard = rocks.Rock(ssnamenr)
name, num = ssocard.name, ssocard.number
name, num

('Thymbraeus', 17365)

## Period estimation

In [5]:
flavor = "SHG1G2"

period_range = (1 / 24, 7)  # 1hour to 7 days

t0 = time.time()
period, chi2red, frequency, power, model, pdf = estimate_synodic_period(
    ssnamenr,
    flavor=flavor,
    Nterms_base=1,
    period_range=period_range,
    return_extra_info=True,
)

In [6]:
print(
    "[{:.2f} seconds] model={}: period={:.2f} hours (chi2red={:.2f}) -- SsODNet: period={:.2f} h ({:s})".format(
        time.time() - t0,
        flavor,
        period,
        chi2red,
        ssocard.spin[0].period.value,
        ",".join(ssocard.spin[0].bibref.bibcode),
    )
)

[13.01 seconds] model=SHG1G2: period=12.67 hours (chi2red=3.06) -- SsODNet: period=12.67 h (2023A&A...675A..24D)


In [12]:
pdf = pdf.sort_values(by="i:jd").reset_index(drop=True)

In [14]:
eph_ec = ssptools.ephemcc( ssnamenr, ep=pdf['i:jd'].to_list(), tcoor=2, rplane=2)

In [15]:
fname = f"{ssnamenr}.obs"

# Reformat for Dima
pdf["na"] = pdf.H * 0.0
cols = [
    "sso_name",
    "i:jd",
    "na",
    "na",
    "residuals",
    "i:sigmapsf",
    "i:fid",
    "na",
    "na",
    "Dobs",
    "Dhelio",
    "Phase",
]
df = pdf[cols]

# Add EC XYZ
df['x'] = eph_ec['px']
df['y'] = eph_ec['py']
df['z'] = eph_ec['pz']


df["sso_name"] = ssnamenr
df.to_string(
    os.path.join("..", "..", "data", "dima", fname),
    header=None,
    index=None,
    formatters={"i:jd": "{:13.6f}".format},
)

/tmp/ipykernel_32446/3245023627.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['x'] = eph_ec['px']
/tmp/ipykernel_32446/3245023627.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['y'] = eph_ec['py']
/tmp/ipykernel_32446/3245023627.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#

In [16]:
eph_ec

,Date,px,py,pz,Dobs,Dhelio,Phase,Elong.,VMag,vx,vy,vz,RV
0,2.458915e+06,-3.157097,-3.998236,-0.570110,5.126225,5.606478,9.289096,114.201089,18.440443,0.009214,0.011773,0.001255,-25.966414
1,2.458915e+06,-3.156323,-3.997245,-0.570004,5.124963,5.606452,9.283180,114.284749,18.439653,0.009190,0.011780,0.001256,-25.949684
2,2.458965e+06,-3.051310,-3.418412,-0.506362,4.610037,5.590282,2.560304,165.629160,17.862889,-0.004536,0.009449,0.001301,-7.180379
3,2.458965e+06,-3.051595,-3.417820,-0.506281,4.609777,5.590261,2.549575,165.690922,17.862033,-0.004551,0.009439,0.001301,-7.148304
4,2.458971e+06,-3.082334,-3.365178,-0.498599,4.590621,5.588247,1.605198,171.062965,17.782851,-0.005862,0.008406,0.001306,-4.101002
...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,2.460695e+06,5.136733,0.202930,0.846719,5.210003,4.855083,10.468838,63.713460,18.210930,0.013705,0.016278,-0.000821,24.262456
288,2.460695e+06,5.136870,0.203092,0.846711,5.210143,4.855083,10.468088,63.705147,18.210959,0.013703,0.016281,-0.000821,24.260778
289,2.460700e+06,5.203037,0.287382,0.842595,5.278650,4.855200,10.070694,59.592368,18.223523,0.012856,0.017568,-0.000832,23.366220
290,2.460700e+06,5.203602,0.288154,0.842559,5.279243,4.855201,10.066977,59.556270,18.223619,0.012848,0.017579,-0.000832,23.357789


In [ ]:
fig, ax = plt.subplots()

ax.scatter(df["i:jd"], pdf["residuals"])